# 06 Resolve Ambiguous Matches with an LLM Tie-Breaker
This notebook reviews only the `AMBIGUOUS` top candidates from the deterministic decision layer. It builds structured LLM prompts, asks the LLM whether each pair is a match, parses the response, and stores cached LLM decisions.

LLM tie-breaker rules:
- Only rank-1 candidates marked `AMBIGUOUS` are sent to the LLM.
- The LLM must return one of `MATCH`, `NO_MATCH`, or `UNCERTAIN`.
- The LLM confidence must be between `0` and `1`.
- In the final decision layer, an LLM `MATCH` or `NO_MATCH` is accepted only when confidence is at least `0.70`.
- If the LLM returns `UNCERTAIN`, fails, or has confidence below `0.70`, the record remains for `REVIEW` or manual inspection.

In [0]:
# Select ambiguous top candidates that need LLM review.
from pyspark.sql import functions as F

source_table = "workspace.entity_resolution_project.company_er_decisions"
llm_input_table = "workspace.entity_resolution_project.company_er_llm_input"
llm_output_table = "workspace.entity_resolution_project.company_er_llm_tiebreaker"

decisions = spark.table(source_table)

ambiguous = (
    decisions
    .filter(
        (F.col("candidate_rank") == 1) &
        (F.col("decision") == "AMBIGUOUS")
    )
    .withColumn(
        "llm_pair_key",
        F.concat_ws(
            "||",
            F.col("left_row_key").cast("string"),
            F.col("right_row_key").cast("string")
        )
    )
)

print("Ambiguous top candidates:", ambiguous.count())
display(ambiguous)

In [0]:
# Exclude ambiguous pairs that already have cached LLM results.
try:
    existing_llm = (
        spark.table(llm_output_table)
        .withColumn(
            "llm_pair_key",
            F.concat_ws(
                "||",
                F.col("left_row_key").cast("string"),
                F.col("right_row_key").cast("string")
            )
        )
        .dropDuplicates(["llm_pair_key"])
    )
except Exception:
    existing_llm = None

if existing_llm is not None:
    new_ambiguous = (
        ambiguous
        .join(
            existing_llm.select("llm_pair_key"),
            on="llm_pair_key",
            how="left_anti"
        )
    )
else:
    new_ambiguous = ambiguous

print("Existing cached LLM rows:", 0 if existing_llm is None else existing_llm.count())
print("New ambiguous pairs sent to LLM:", new_ambiguous.count())

In [0]:
# Build LLM judge prompts and save the LLM input table.
llm_input = (
    new_ambiguous
    .withColumn(
        "llm_prompt",
        F.concat_ws(
            "\n",
            F.lit("You are an entity resolution judge."),
            F.lit("Company A is the noisy procurement input record."),
            F.lit("Company B is the enriched candidate company record."),
            F.lit("Decide whether Company A and Company B refer to the same real-world company."),
            F.lit("Return ONLY valid JSON with keys: decision, confidence, reason."),
            F.lit("decision must be one of: MATCH, NO_MATCH, UNCERTAIN."),
            F.lit("confidence must be a number between 0 and 1."),
            F.lit("reason must be maximum 50 words."),
            F.lit("Do not include markdown, bullet points, or extra text."),
            F.lit(""),
            F.concat(F.lit("Company A name: "), F.coalesce(F.col("left_company_name"), F.lit(""))),
            F.concat(F.lit("Company B name: "), F.coalesce(F.col("right_company_name"), F.lit(""))),
            F.concat(F.lit("Company A country: "), F.coalesce(F.col("left_country"), F.lit(""))),
            F.concat(F.lit("Company A country code: "), F.coalesce(F.col("left_country_code"), F.lit(""))),
            F.concat(F.lit("Company B country: "), F.coalesce(F.col("right_country"), F.lit(""))),
            F.concat(F.lit("Company B country code: "), F.coalesce(F.col("right_country_code"), F.lit(""))),
            F.concat(F.lit("Company A city: "), F.coalesce(F.col("left_city"), F.lit(""))),
            F.concat(F.lit("Company B city: "), F.coalesce(F.col("right_city"), F.lit(""))),
            F.concat(F.lit("Company B website domain: "), F.coalesce(F.col("right_website_domain"), F.lit(""))),
            F.concat(F.lit("Name similarity: "), F.round(F.col("name_similarity"), 3).cast("string")),
            F.concat(F.lit("Semantic similarity: "), F.coalesce(F.round(F.col("semantic_similarity"), 3).cast("string"), F.lit(""))),
            F.concat(F.lit("Entropy norm: "), F.coalesce(F.round(F.col("entropy_norm"), 3).cast("string"), F.lit(""))),
            F.concat(F.lit("Composite score: "), F.round(F.col("composite_score"), 3).cast("string")),
            F.concat(F.lit("Top1 score: "), F.round(F.col("top1_score"), 3).cast("string")),
            F.concat(F.lit("Top2 score: "), F.round(F.col("top2_score"), 3).cast("string")),
            F.concat(F.lit("Score gap: "), F.round(F.col("score_gap_top1_top2"), 3).cast("string")),
            F.lit(""),
            F.lit("Company A text:"),
            F.coalesce(F.col("left_search_text"), F.lit("")),
            F.lit(""),
            F.lit("Company B text:"),
            F.coalesce(F.col("right_search_text"), F.lit(""))
        )
    )
)

(
    llm_input.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(llm_input_table)
)

display(
    llm_input.select(
        "left_company_name",
        "right_company_name",
        "top1_score",
        "score_gap_top1_top2",
        "llm_prompt"
    )
)

In [0]:
# Run the LLM tie-breaker on new ambiguous pairs.
llm_sample = spark.table(llm_input_table)

llm_results = llm_sample.selectExpr(
    "*",
    """
    ai_query(
      'databricks-gpt-oss-120b',
      llm_prompt,
      modelParameters => named_struct('max_tokens', 400, 'temperature', 0.0),
      failOnError => false
    ) AS llm_raw_response
    """
)

display(llm_results.select("left_company_name", "right_company_name", "llm_raw_response"))

In [0]:
# Parse LLM responses and merge them with cached results.
parsed = (
    llm_results
    .withColumn("llm_response_text", F.col("llm_raw_response.result"))
    .withColumn("llm_error_message", F.col("llm_raw_response.errorMessage"))
    .withColumn(
        "llm_response_clean",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col("llm_response_text"),
                        r"^```json\s*",
                        ""
                    ),
                    r"^```\s*",
                    ""
                ),
                r"\s*```$",
                ""
            )
        )
    )
    .withColumn(
        "llm_decision",
        F.regexp_extract(
            F.col("llm_response_clean"),
            r'"decision"\s*:\s*"([^"]+)"',
            1
        )
    )
    .withColumn(
        "llm_confidence_text",
        F.regexp_extract(
            F.col("llm_response_clean"),
            r'"confidence"\s*:\s*([0-9.]+)',
            1
        )
    )
    .withColumn(
        "llm_confidence",
        F.expr("try_cast(llm_confidence_text AS DOUBLE)")
    )
    .withColumn(
        "llm_reason_closed",
        F.regexp_extract(
            F.col("llm_response_clean"),
            r'"reason"\s*:\s*"((?:[^"\\\\]|\\\\.)*)"',
            1
        )
    )
    .withColumn(
        "llm_reason_open",
        F.regexp_extract(
            F.col("llm_response_clean"),
            r'"reason"\s*:\s*"(.*)$',
            1
        )
    )
    .withColumn(
        "llm_reason",
        F.when(F.col("llm_reason_closed") != "", F.col("llm_reason_closed"))
         .when(F.col("llm_reason_open") != "", F.col("llm_reason_open"))
         .otherwise(None)
    )
    .withColumn(
        "llm_reason",
        F.regexp_replace(F.col("llm_reason"), r'\\"', '"')
    )
    .withColumn(
        "llm_decision",
        F.when(F.col("llm_decision") == "", None).otherwise(F.col("llm_decision"))
    )
    .withColumn(
        "llm_reason",
        F.when(F.col("llm_reason") == "", None).otherwise(F.col("llm_reason"))
    )
)

if existing_llm is not None:
    final_llm = (
        existing_llm
        .unionByName(parsed, allowMissingColumns=True)
        .dropDuplicates(["llm_pair_key"])
    )
else:
    final_llm = parsed

(
    final_llm.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(llm_output_table)
)

display(
    final_llm.select(
        "left_company_name",
        "right_company_name",
        "llm_decision",
        "llm_confidence",
        "llm_reason",
        "llm_error_message",
        "llm_response_clean"
    )
)

In [0]:
# Validate cached LLM output quality and error counts.
display(
    final_llm.select(
        F.count("*").alias("total_cached_llm_rows"),
        F.sum(F.col("llm_decision").isNull().cast("int")).alias("null_decision"),
        F.sum(F.col("llm_reason").isNull().cast("int")).alias("null_reason"),
        F.sum(F.col("llm_error_message").isNotNull().cast("int")).alias("error_rows")
    )
)